In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")

In [ ]:
insurance_claims = pd.read_parquet("../data/clean_insurance_claims.parquet")

In [ ]:
# remove columns that are not useful for analysis or modelling
insurance_claims = insurance_claims.drop(columns = ["policy_number", "insured_zip", "incident_location", "fraud_reported"])

In [ ]:
numeric_df = insurance_claims.select_dtypes(include = ["int64", "float64"])
categorical_df = insurance_claims.select_dtypes(include = ["object"])
datetime_df = insurance_claims.select_dtypes(include = ["datetime64[ns]"])

categorical_df["total_claim_amount"] = insurance_claims["total_claim_amount"]
datetime_df["total_claim_amount"] = insurance_claims["total_claim_amount"]

#### TARGET
- The target feature is bimodal.
- Around 180 claims fall around 10000, while the remaining claims concentrate between 20000 and 120000.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (12, 4), constrained_layout = True)
axes = axes.flatten()
plt.suptitle("Distribution of Total Claim Amount")

sns.histplot(data = insurance_claims, x = "total_claim_amount", kde = True, ax = axes[0])
sns.boxplot(data = insurance_claims, x = "total_claim_amount", ax = axes[1])

plt.show()

#### NUMERIC
1. In the scatter plots, there is a strong relationship between the 'total_claim_amount' and the injury, property, and vehicle claims, which is expected as the features directly contribute to the total claim. There is a clear positive relationship between the 'number_of_vehicles_involved' and  'incident_hour_of_the_day' and the 'total_claim_amount'. Apart from this, the 'months_as_customer' and 'age' show a weak positive relationship with the 'total_claim_amount'. All other features do not portray any linear relationship with the target. 

2. Observing the correlation map, the same conclusions can be gathered as from the scatter plot. The claim features are highly correlated, while 'number_of_vehicles_involved' and 'incident_hour_of_the_day' reveal a weak positive relationship with the target. The 'months_as_customer' and 'age' features have an extremely weak positive correlation with the 'total_claim_amount'.

In [ ]:
dots = {
    "alpha": 0.7,
    "s": 15
}
line = {
    "color": "red",
    "linewidth": 2
}


fig, axes = plt.subplots(4, 4, figsize = (12, 12), constrained_layout = True)
axes = axes.flatten() 

for ax, column in zip(axes, numeric_df.columns):
    sns.regplot(data = numeric_df, y = "total_claim_amount", x = column, scatter_kws = dots, line_kws = line, ax = ax)
    ax.set_title(column, fontweight = "semibold")

plt.show()

In [ ]:
plt.figure(figsize = (14, 10), constrained_layout = True)

sns.heatmap(numeric_df.corr(), annot = True, fmt = ".2g", linewidths = 0.5, cmap = "coolwarm", vmin = -1)
plt.title("Correlation of Numeric Features", fontweight = "semibold")

plt.show()

#### CATEGORICAL FEATURES
Analysing the boxplots, we see a number of relationships between the features and 'total_claim_amount'. In the 'insured_education_level' feature, the 'MD', 'PhD', and 'College' variables have most of the data within the second modal range, 20000-120000, with possible outliers in the first modal range. This pattern is seen frequently in the features:
- 'insured_occupation': 'machine-op-inspct', 'armed-forces', 'priv-house-serv', 'protective-serv', 'transport-moving', and 'handlers-cleaners'
- 'insured_hobbies': 'sleeping', 'camping', 'hiking', 'chess', 'basketball', 'cross-fit', and 'exercise'
- 'incident_state': 'SC', 'NY', and 'PA'
- 'incident_city': 'Columbus'
- 'police_report_available': 'unknown'
- 'auto_make': 'Saab', 'Nissan', 'Ford', and 'Honda'
- 'incident_type': 'Single Vehicle Collision' and 'Multi-vehicle Collision'
- 'collision_type': 'Side Collision', 'Rear Collision', and 'Front Collision'
- 'incident_severity': 'Major Damage' and 'Total Loss'
- 'authorities_contacted': 'Fire', 'Other', and 'Ambulance'

Another observation is variables that only appear within the first modal range. These variables have no 'total_claim_amount' above 20000, including outliers. The variables here are:
- 'incident_type': 'Vehicle Theft' and 'Parked Car'
- 'collision_type': 'unknown'
- 'incident_severity': 'Trivial Damage'
- 'authorities_contacted': 'unknown'

In [ ]:
categorical_names = categorical_df.select_dtypes(include = ["object"]).columns

fig, axes = plt.subplots(4, 4, figsize = (16, 16), constrained_layout = True)
axes = axes.flatten() 

for ax, column in zip(axes, categorical_names):
    sns.boxplot(data = categorical_df, x = "total_claim_amount", y = column, ax = ax)
    ax.set_title(column, fontweight = "semibold")

plt.show()

#### DATETIME FEATURES
In the line chart, the 'policy_bind_date_month' shows a decrease in the 'total_claim_amount' during March, potentially suggesting a relationship. However, this pattern is most likely a result of a sampling issue or an event during March that affected the total claim, rather than a causal relationship.

Inspecting the boxplots, the 'policy_bind_date_month' March appears more spread towards the lower claims, reaffirming the line chart observations.  In the 'incident_date_month', March appears to reflect the second modal range. This is not a pattern but a sampling problem, as the total March incidents equal 12, 1.2% of the total dataset.

In [ ]:
datetime_df["policy_bind_date_month"] = datetime_df["policy_bind_date"].dt.month
datetime_df["policy_bind_date_day"] = datetime_df["policy_bind_date"].dt.day
datetime_df["incident_date_month"] = datetime_df["incident_date"].dt.month
datetime_df["incident_date_day"] = datetime_df["incident_date"].dt.day

policy_fraud_rate_month = datetime_df.groupby(["policy_bind_date_month"]).mean().reset_index()
policy_fraud_rate_day = datetime_df.groupby(["policy_bind_date_day"]).mean().reset_index()
incident_fraud_rate_month = datetime_df.groupby(["incident_date_month"]).mean().reset_index()
incident_fraud_rate_day = datetime_df.groupby(["incident_date_day"]).mean().reset_index()

date_column = [(policy_fraud_rate_month, "policy_bind_date_month"), (policy_fraud_rate_day, "policy_bind_date_day"), 
               (incident_fraud_rate_month, "incident_date_month"), (incident_fraud_rate_day, "incident_date_day")]

fig, axes = plt.subplots(2, 2, figsize = (10, 5), constrained_layout = True)
axes = axes.flatten()

for ax, (data, column) in zip(axes, date_column):
    sns.lineplot(data = data, x = column, y = "total_claim_amount", ax = ax, legend = False)
    ax.set_title(column, fontweight = "semibold")

plt.show()

In [ ]:
datetime_names = datetime_df.select_dtypes(include = ["int32"]).columns

fig, axes = plt.subplots(2, 2, figsize = (12, 6), constrained_layout = True)
axes = axes.flatten() 

for ax, column in zip(axes, datetime_names):
    sns.boxplot(data = datetime_df, x = column, y = "total_claim_amount", ax = ax)
    ax.set_title(column, fontweight = "semibold")

plt.show()